Детекция бота

Для каждой `cookie_id` оценивается вероятность того, что события относятся к боту. В качестве основной метрики используется Precision при Recall ≥ 70%. Разделение на обучение и проверку выполнено по времени: проверочные дни всегда идут после обучающих.


In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

from metric import precision_at_recall

DATA_DIR = Path("data")
train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
events = pd.read_csv(DATA_DIR / "events.csv.gz", parse_dates=["event_ts"])

assert train.cookie_id.is_unique and test.cookie_id.is_unique
assert set(train.cookie_id).isdisjoint(test.cookie_id)
print("train / test / events:", train.shape, test.shape, events.shape)
print("Доля положительного класса:", round(train.target.mean(), 4))


train / test / events: (11091, 5) (4909, 4) (328905, 14)
Доля положительного класса: 0.0811


Подготовка событий

Из истории куки берём события с `window_start_ts <= event_ts < window_end_ts`, чтобы выкинуть события, не подходящие под временной промежуток.
Полные дубликаты удаляем до построения признаков. В `platform` различия регистра и обозначения `desktop`/`iphone` сводим к трём категориям. Такой порядок обработки выбран по результатам сравнения на временной валидации.


In [2]:
def select_window(events, meta):
    joined = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )
    inside = joined.event_ts.ge(joined.window_start_ts) & joined.event_ts.lt(joined.window_end_ts)
    return joined.loc[inside].copy()


def prepare_events(events):
    prepared = events.drop_duplicates().copy()
    prepared["platform"] = (
        prepared.platform.str.strip().str.lower()
        .replace({"desktop": "web", "iphone": "ios"})
    )
    return prepared.sort_values(["cookie_id", "event_ts"], kind="stable").reset_index(drop=True)


raw_train_events = select_window(events, train)
raw_test_events = select_window(events, test)
train_events = prepare_events(raw_train_events)
test_events = prepare_events(raw_test_events)

assert set(train_events.cookie_id) == set(train.cookie_id)
assert set(test_events.cookie_id) == set(test.cookie_id)
print("События в окне (train / test):", len(raw_train_events), len(raw_test_events))
print("Удалено точных дублей:", len(raw_train_events) - len(train_events),
      len(raw_test_events) - len(test_events))


События в окне (train / test): 198436 89690
Удалено точных дублей: 2952 1341


Признаки

Одна строка признаков соответствует одной куки. Первая группа описывает объём и состав действий; вторая — интервалы и длительность активности; третья — разнообразие объявлений и поисков; четвертая — координаты курсора и платформу. Пропуски, при которых статистика не определена, обозначаются `-1`; нули оставляем нулями


In [3]:
EVENT_TYPES = [
    "item_view",
    "search_results_view",
    "photo_swipe",
    "favorite_add",
    "seller_page_view",
    "contact_phone_show",
    "captcha_shown",
    "login",
    "contact_chat_open",
    "contact_message_sent",
]

def build_behavior_features(ev, meta):
    cookie_index = pd.Index(meta["cookie_id"], name="cookie_id")
    features = pd.DataFrame(index=cookie_index)

    grouped = ev.groupby("cookie_id")

    features["n_events"] = grouped.size().reindex(
        cookie_index, fill_value=0
    )
    features["item_nunique"] = grouped["item_id"].nunique().reindex(
        cookie_index, fill_value=0
    )

    counts = pd.crosstab(ev["cookie_id"], ev["event_name"])
    counts = counts.reindex(
        index=cookie_index,
        columns=EVENT_TYPES,
        fill_value=0,
    )

    total = features["n_events"].replace(0, np.nan)

    for event in EVENT_TYPES:
        features[f"count_{event}"] = counts[event]
        features[f"share_{event}"] = (
            counts[event].div(total).fillna(0)
        )

    return features


In [4]:
def build_time_features(ev, meta):
    cookie_index = pd.Index(meta["cookie_id"], name="cookie_id")

    ordered = ev.sort_values(
        ["cookie_id", "event_ts"],
        kind="stable",
    ).copy()

    grouped = ordered.groupby("cookie_id")

    ordered["gap_seconds"] = (
        grouped["event_ts"].diff().dt.total_seconds()
    )

    features = pd.DataFrame(index=cookie_index)

    first_event = grouped["event_ts"].min()
    last_event = grouped["event_ts"].max()

    features["activity_span_seconds"] = (
        last_event - first_event
    ).dt.total_seconds()


    ordered["hour_bucket"] = ordered["event_ts"].dt.floor("h")
    features["active_hours"] = (
        ordered.groupby("cookie_id")["hour_bucket"].nunique()
    )

    gaps = ordered.groupby("cookie_id")["gap_seconds"]

    features["gap_count"] = gaps.count()
    features["gap_mean"] = gaps.mean()
    features["gap_median"] = gaps.median()
    features["gap_std"] = gaps.std(ddof=0)
    features["gap_min"] = gaps.min()
    features["gap_max"] = gaps.max()
    features["gap_q25"] = gaps.quantile(0.25)
    features["gap_q75"] = gaps.quantile(0.75)


    features["gap_relative_std"] = (
        features["gap_std"]
        / features["gap_mean"].replace(0, np.nan)
    )


    valid_gaps = ordered.loc[
        ordered["gap_seconds"].notna(),
        ["cookie_id", "gap_seconds"],
    ].copy()

    for seconds in [0, 1, 5, 10, 60]:
        valid_gaps["is_short"] = (
            valid_gaps["gap_seconds"] <= seconds
        )
        features[f"gap_share_le_{seconds}s"] = (
            valid_gaps.groupby("cookie_id")["is_short"].mean()
        )

    long_pauses = (
        ordered["gap_seconds"].gt(30 * 60)
        .groupby(ordered["cookie_id"])
        .sum()
    )

    event_counts = grouped.size().reindex(cookie_index, fill_value=0)

    features["session_count_30min"] = (
        long_pauses.reindex(cookie_index, fill_value=0)
        + event_counts.gt(0).astype(int)
    )


    features["has_no_gaps"] = (
        features["gap_count"].fillna(0).eq(0).astype(int)
    )

    return features.replace([np.inf, -np.inf], np.nan).fillna(-1)


In [5]:
def build_diversity_features(ev, meta):
    cookie_index = pd.Index(meta["cookie_id"], name="cookie_id")
    features = pd.DataFrame(index=cookie_index)

    data = ev.copy()

    data["query_normalized"] = (
        data["search_query"]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )

    diversity_columns = {
        "item_id": "item",
        "item_category": "category",
        "item_location": "location",
        "query_normalized": "query",
        "event_name": "event_type",
    }

    for column, prefix in diversity_columns.items():
        grouped = data.groupby("cookie_id")[column]

        nonmissing_count = grouped.count().reindex(
            cookie_index, fill_value=0
        )
        unique_count = grouped.nunique().reindex(
            cookie_index, fill_value=0
        )

        value_counts = (
            data.dropna(subset=[column])
            .groupby(["cookie_id", column])
            .size()
        )
        top_count = (
            value_counts.groupby(level=0)
            .max()
            .reindex(cookie_index, fill_value=0)
        )

        denominator = nonmissing_count.replace(0, np.nan)

        if prefix != "item":
            features[f"{prefix}_nunique"] = unique_count

        features[f"{prefix}_unique_share"] = (
            unique_count / denominator
        )
        features[f"{prefix}_top_share"] = (
            top_count / denominator
        )
        
    views = data.loc[
        data["event_name"].eq("item_view") & data["item_id"].notna()
    ]
    view_groups = views.groupby("cookie_id")["item_id"]

    view_count = view_groups.count().reindex(cookie_index, fill_value=0)
    viewed_items = view_groups.nunique().reindex(
        cookie_index, fill_value=0
    )

    features["repeat_item_view_share"] = (
        1 - viewed_items / view_count.replace(0, np.nan)
    )

    search_events = data.loc[
        data["event_name"].eq("search_results_view")
    ]
    pages = search_events.groupby("cookie_id")["search_page"]

    features["search_page_max"] = pages.max()
    features["search_page_mean"] = pages.mean()
    features["search_page_nunique"] = pages.nunique().reindex(
        cookie_index, fill_value=0
    )

    return features.replace([np.inf, -np.inf], np.nan).fillna(-1)


In [6]:
def build_pointer_features(ev, meta):
    cookie_index = pd.Index(meta["cookie_id"], name="cookie_id")
    features = pd.DataFrame(index=cookie_index)

    data = ev.copy()

    data["has_pointer"] = (
        data["pointer_x"].notna() & data["pointer_y"].notna()
    )

    grouped = data.groupby("cookie_id")

    features["pointer_count"] = (
        grouped["has_pointer"].sum()
        .reindex(cookie_index, fill_value=0)
    )
    features["pointer_share"] = grouped["has_pointer"].mean()

    points = data.loc[data["has_pointer"]].copy()
    point_groups = points.groupby("cookie_id")

    for coordinate in ["pointer_x", "pointer_y"]:
        values = point_groups[coordinate]

        features[f"{coordinate}_std"] = values.std(ddof=0)
        features[f"{coordinate}_range"] = values.max() - values.min()

    unique_points = (
        points.drop_duplicates(["cookie_id", "pointer_x", "pointer_y"])
        .groupby("cookie_id")
        .size()
        .reindex(cookie_index, fill_value=0)
    )

    features["pointer_unique_pairs"] = unique_points

    pointer_denominator = features["pointer_count"].replace(0, np.nan)
    features["pointer_unique_share"] = (
        unique_points / pointer_denominator
    )

    pair_counts = (
        points.groupby(["cookie_id", "pointer_x", "pointer_y"])
        .size()
    )
    most_common_pair = (
        pair_counts.groupby(level=0)
        .max()
        .reindex(cookie_index, fill_value=0)
    )
    features["pointer_top_pair_share"] = (
        most_common_pair / pointer_denominator
    )

    platform_counts = pd.crosstab(
        data["cookie_id"], data["platform"]
    ).reindex(
        index=cookie_index,
        columns=["web", "android", "ios"],
        fill_value=0,
    )

    event_count = grouped.size().reindex(cookie_index, fill_value=0)
    denominator = event_count.replace(0, np.nan)

    for platform in ["web", "android", "ios"]:
        features[f"platform_share_{platform}"] = (
            platform_counts[platform] / denominator
        )

    return features.replace([np.inf, -np.inf], np.nan).fillna(-1)


In [7]:
def build_features(ev, meta):
    return (
        build_behavior_features(ev, meta)
        .join(build_time_features(ev, meta))
        .join(build_diversity_features(ev, meta))
        .join(build_pointer_features(ev, meta))
    )


X_train = build_features(train_events, train)
X_test = build_features(test_events, test)
y = train.target.to_numpy()

assert X_train.index.equals(pd.Index(train.cookie_id))
assert X_test.index.equals(pd.Index(test.cookie_id))
assert X_train.columns.equals(X_test.columns) and X_train.columns.is_unique
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_test.to_numpy()).all()
print("Таблицы признаков:", X_train.shape, X_test.shape)


Таблицы признаков: (11091, 70) (4909, 70)


Проверка по времени

В `quickstart` использовались два признака и Random Forest. Сравниваем его результат на 17–19 апреля с выбранной моделью; для последней также приводим три последовательных временных проверки. В каждом разбиении признаки проверочных куки рассчитаны только по событиям их собственного окна, а модель обучается на более ранних днях.


In [8]:
raw_counts = raw_train_events.groupby("cookie_id")
baseline_X = pd.DataFrame(index=X_train.index)
baseline_X["n_events"] = raw_counts.size().reindex(baseline_X.index, fill_value=0)
baseline_X["item_nunique"] = raw_counts.item_id.nunique().reindex(baseline_X.index, fill_value=0)

last_fold = train.window_start_ts.ge("2026-04-17").to_numpy()
baseline = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=3, random_state=0, n_jobs=-1
)
baseline.fit(baseline_X.loc[~last_fold], y[~last_fold])
baseline_pred = baseline.predict_proba(baseline_X.loc[last_fold])[:, 1]
baseline_score = precision_at_recall(y[last_fold], baseline_pred)
print("Quickstart, 17–19 апреля:", round(baseline_score, 4))


Quickstart, 17–19 апреля: 0.1025


In [9]:
FOLDS = [
    ("2026-04-11", "2026-04-14"),
    ("2026-04-14", "2026-04-17"),
    ("2026-04-17", "2026-04-20"),
]


def make_model():
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=5.0,
        early_stopping=False,
        random_state=0,
    )


validation_rows = []
for start, end in FOLDS:
    fit_mask = train.window_start_ts.lt(start).to_numpy()
    check_mask = train.window_start_ts.ge(start).to_numpy() & train.window_start_ts.lt(end).to_numpy()
    model = make_model()
    model.fit(X_train.loc[fit_mask], y[fit_mask])
    scores = model.predict_proba(X_train.loc[check_mask])[:, 1]
    validation_rows.append({
        "Проверка": f"{pd.Timestamp(start):%d.%m}–{(pd.Timestamp(end)-pd.Timedelta(days=1)):%d.%m}",
        "Куки": int(check_mask.sum()),
        "Боты": int(y[check_mask].sum()),
        "P@R≥0.7": precision_at_recall(y[check_mask], scores),
    })

validation = pd.DataFrame(validation_rows)
print(validation.round(4).to_string(index=False))
print("Среднее P@R≥0.7:", round(validation["P@R≥0.7"].mean(), 4))


   Проверка  Куки  Боты  P@R≥0.7
11.04–13.04  2556   199   0.6965
14.04–16.04  2367   195   0.6816
17.04–19.04  1951   160   0.7600
Среднее P@R≥0.7: 0.7127


Текущий набор признаков и параметры модели были выбраны по этой временной валидации. Также попробоавли добавить такие метрики как, браузер пользователя, как давно появилось куки, насколько часто совершается какое-либо дейтвие. Но тк эти варианты не дали никакого значительного улучшения поэтому в итоговом решении оставили выбранные 70 признаков и градиентный бустинг.

Финальное обучение и ответ

После выбора модели используем все размеченные куки. В файл записывается оценка класса `1`; порог выбирает проверяющая система. Порядок строк берётся из `sample_submission.csv`.


In [10]:
final_model = make_model()
final_model.fit(X_train, y)
bot_column = np.flatnonzero(final_model.classes_ == 1).item()

predictions = pd.DataFrame({
    "cookie_id": X_test.index.to_numpy(),
    "score": final_model.predict_proba(X_test)[:, bot_column],
})
sample = pd.read_csv("sample_submission.csv", dtype={"cookie_id": str})
submission = sample[["cookie_id"]].merge(
    predictions, on="cookie_id", how="left", sort=False, validate="one_to_one"
)

assert submission.columns.tolist() == ["cookie_id", "score"]
assert submission.cookie_id.is_unique and len(submission) == len(test)
assert set(submission.cookie_id) == set(test.cookie_id)
assert submission.score.notna().all() and submission.score.between(0, 1).all()
submission.to_csv("submission.csv", index=False)
print("submission.csv:", submission.shape)
print(submission.score.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(6))


submission.csv: (4909, 2)
count    4909.000000
mean        0.086100
std         0.213681
min         0.000279
50%         0.011324
90%         0.214108
95%         0.725698
99%         0.986495
max         0.997378
Name: score, dtype: float64


In [11]:
requirements = "\n".join(
    f"{name}=={version(name)}" for name in ("numpy", "pandas", "scikit-learn")
)
Path("requirements.txt").write_text(requirements + "\n", encoding="utf-8")
Path("python_version.txt").write_text(sys.version.split()[0] + "\n", encoding="utf-8")
print("Python:", sys.version.split()[0])
print(requirements)


Python: 3.13.9
numpy==2.3.5
pandas==2.3.3
scikit-learn==1.7.2
